# Persistencia en MongoDB y caché en Redis
Valentina Montenegro · PC4

Se guardan y consultan resultados de K-Means desde Go y se comprueba la caché con Redis.

## 1. Configuración del entorno
Se instalan Go, MongoDB y Redis. Los servicios funcionan dentro de Colab; sus datos se pierden al eliminar el entorno. Las salidas quedan guardadas en el cuaderno.

In [ ]:
import os, sys, subprocess, json, time, socket, hashlib, platform, uuid
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

WORK = Path('/content/pc4_persistencia')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

def run(args, **kwargs):
    result = subprocess.run(args, text=True, capture_output=True, **kwargs)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.returncode:
        print(result.stderr[-8000:])
        result.check_returncode()
    return result.stdout

release = dict(line.split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
codename = release.get('VERSION_CODENAME','').strip('"')
assert codename in ('jammy','noble','focal'), f'Ubuntu no previsto: {codename}; no continuar con un repositorio distinto.'
assert platform.machine() == 'x86_64', 'Este cuaderno está preparado para CPU x86_64.'
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','-qq','curl','gnupg','golang-go','gcc','redis-server'])
key = subprocess.check_output(['curl','-fsSL','https://pgp.mongodb.com/server-8.0.asc'])
subprocess.run(['gpg','--batch','--yes','--dearmor','-o','/usr/share/keyrings/pc4-mongodb-8.0.gpg'],input=key,check=True)
Path('/etc/apt/sources.list.d/pc4-mongodb-8.0.list').write_text(f'deb [arch=amd64 signed-by=/usr/share/keyrings/pc4-mongodb-8.0.gpg] https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse\n')
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','-qq','mongodb-org-server'])

def port_open(port):
    with socket.socket() as s:
        s.settimeout(.3)
        return s.connect_ex(('127.0.0.1',port)) == 0

DBPATH = WORK/'mongo_data'
DBPATH.mkdir(exist_ok=True)
if not port_open(27018):
    run(['mongod','--dbpath',str(DBPATH),'--bind_ip','127.0.0.1','--port','27018','--logpath',str(WORK/'mongo.log'),'--fork'])
if not port_open(6380):
    run(['redis-server','--bind','127.0.0.1','--port','6380','--protected-mode','yes','--daemonize','yes','--save','','--appendonly','no','--logfile',str(WORK/'redis.log')])
for _ in range(40):
    if port_open(27018) and port_open(6380): break
    time.sleep(.25)
assert port_open(27018) and port_open(6380), 'Revisar mongo.log y redis.log.'

os.environ['MONGODB_URI']='mongodb://127.0.0.1:27018'
os.environ['MONGODB_DATABASE']='eas_pc4_'+uuid.uuid4().hex[:10]
os.environ['REDIS_ADDR']='127.0.0.1:6380'
os.environ['REDIS_PASSWORD']=''
os.environ['CACHE_TTL_SECONDS']='3'
os.environ['GOMAXPROCS']='2'
os.environ['CGO_ENABLED']='1'
os.environ['PC4_INTEGRATION']='1'
ENVIRONMENT={'fecha_utc':datetime.now(timezone.utc).isoformat(),'sistema':platform.platform(),'cpu_logicas':os.cpu_count(),'GOMAXPROCS':os.environ['GOMAXPROCS'],'database':os.environ['MONGODB_DATABASE']}
for name,args in [('go',['go','version']),('mongodb',['mongod','--version']),('redis',['redis-server','--version']),('cpu',['getconf','_NPROCESSORS_ONLN']),('memoria',['free','-h'])]:
    ENVIRONMENT[name]=run(args)
print('Servicios preparados. Base aislada de esta ejecución:', os.environ['MONGODB_DATABASE'])

## 2. Entidades e índices — `modelos.go`
`Dataset` identifica el archivo utilizado. `Trabajo` guarda parámetros, estado y resultado, y se relaciona con el dataset mediante `dataset_id`.

MongoDB utiliza identificadores únicos e índices por dataset, estado y fecha. El resultado se guarda dentro del trabajo para actualizar ambos juntos.

In [ ]:
%%writefile /content/pc4_persistencia/go.mod
module eas-persistencia-pc4

go 1.22

require (
	github.com/redis/go-redis/v9 v9.7.0
	go.mongodb.org/mongo-driver/v2 v2.1.0
)

require (
	github.com/cespare/xxhash/v2 v2.2.0 // indirect
	github.com/dgryski/go-rendezvous v0.0.0-20200823014737-9f7001d12a5f // indirect
	github.com/golang/snappy v0.0.4 // indirect
	github.com/klauspost/compress v1.16.7 // indirect
	github.com/xdg-go/pbkdf2 v1.0.0 // indirect
	github.com/xdg-go/scram v1.1.2 // indirect
	github.com/xdg-go/stringprep v1.0.4 // indirect
	github.com/youmark/pkcs8 v0.0.0-20240726163527-a2c0da244d78 // indirect
	golang.org/x/crypto v0.33.0 // indirect
	golang.org/x/sync v0.11.0 // indirect
	golang.org/x/text v0.22.0 // indirect
)

In [ ]:
%%writefile /content/pc4_persistencia/modelos.go
package main

import (
	"encoding/hex"
	"errors"
	"math"
	"time"
)

var (
	ErrNoEncontrado = errors.New("identificador inexistente")
	ErrEstado       = errors.New("transición de estado no permitida")
	ErrPendiente    = errors.New("la ejecución todavía no tiene un resultado definitivo")
)

// Dataset conserva procedencia; los CSV permanecen en sus archivos de origen.
type Dataset struct {
	ID       string `bson:"_id" json:"dataset_id"`
	Archivo  string `bson:"archivo" json:"archivo"`
	Fuente   string `bson:"fuente" json:"fuente"`
	SHA256   string `bson:"sha256" json:"sha256"`
	Periodo  string `bson:"periodo" json:"periodo"`
	Vectores int    `bson:"vectores" json:"vectores"`
}

type Parametros struct {
	K          int     `bson:"k" json:"k"`
	Semilla    int64   `bson:"semilla" json:"semilla"`
	Tolerancia float64 `bson:"tolerancia" json:"tolerancia"`
	MaxIter    int     `bson:"max_iter" json:"max_iter"`
}

type Resultado struct {
	Centroides    [][]float64 `bson:"centroides" json:"centroides"`
	Conteos       []int       `bson:"conteos" json:"conteos"`
	Iteraciones   int         `bson:"iteraciones" json:"iteraciones"`
	Inercia       float64     `bson:"inercia" json:"inercia"`
	Convergio     bool        `bson:"convergio" json:"convergio"`
	SegundosCarga float64     `bson:"segundos_carga" json:"segundos_carga"`
	SegundosML    float64     `bson:"segundos_ml" json:"segundos_ml"`
	Origen        string      `bson:"origen" json:"origen"`
}

// El resultado resumido se incrusta en jobs: estado y resultado se guardan
// en una única escritura atómica. No se incrustan las N asignaciones.
type Trabajo struct {
	ID          string     `bson:"_id" json:"job_id"`
	DatasetID   string     `bson:"dataset_id" json:"dataset_id"`
	Parametros  Parametros `bson:"parametros" json:"parametros"`
	Estado      string     `bson:"estado" json:"estado"`
	Creado      time.Time  `bson:"creado" json:"creado"`
	Actualizado time.Time  `bson:"actualizado" json:"actualizado"`
	Resultado   *Resultado `bson:"resultado,omitempty" json:"resultado,omitempty"`
	Error       string     `bson:"error,omitempty" json:"error,omitempty"`
}

func finito(x float64) bool { return !math.IsNaN(x) && !math.IsInf(x, 0) }
func validarDataset(d Dataset) error {
	b, err := hex.DecodeString(d.SHA256)
	if d.ID == "" || d.Archivo == "" || d.Fuente == "" || err != nil || len(b) != 32 || d.Vectores < 1 || (d.Periodo != "1_hora" && d.Periodo != "24_horas") {
		return errors.New("dataset inválido: revise procedencia, hash, periodo y vectores")
	}
	return nil
}
func validarParametros(p Parametros, n int) error {
	if p.K < 1 || p.K > n || p.MaxIter < 1 || !finito(p.Tolerancia) || p.Tolerancia < 0 {
		return errors.New("parámetros inválidos")
	}
	return nil
}
func validarResultado(r Resultado, p Parametros, n int) error {
	if len(r.Centroides) != p.K || len(r.Conteos) != p.K || r.Iteraciones < 1 || r.Iteraciones > p.MaxIter || r.Origen == "" {
		return errors.New("dimensiones o metadatos del resultado inválidos")
	}
	if !finito(r.Inercia) || r.Inercia < 0 || !finito(r.SegundosCarga) || r.SegundosCarga < 0 || !finito(r.SegundosML) || r.SegundosML < 0 {
		return errors.New("métricas inválidas")
	}
	suma := 0
	for i, c := range r.Centroides {
		if len(c) != 2 || !finito(c[0]) || !finito(c[1]) || r.Conteos[i] < 0 || r.Conteos[i] > n-suma {
			return errors.New("centroides o conteos inválidos")
		}
		suma += r.Conteos[i]
	}
	if suma != n {
		return errors.New("la suma de conteos no coincide con el dataset")
	}
	return nil
}

## 3. Persistencia — `mongodb.go`
Se crean las colecciones e índices, se registran trabajos y se consultan por `job_id`. Los estados son `queued`, `running`, `completed` y `failed`. MongoDB conserva el resultado aunque se elimine su copia en Redis.

In [ ]:
%%writefile /content/pc4_persistencia/mongodb.go
package main

import (
	"context"
	"errors"
	"fmt"
	"time"

	"go.mongodb.org/mongo-driver/v2/bson"
	"go.mongodb.org/mongo-driver/v2/mongo"
	"go.mongodb.org/mongo-driver/v2/mongo/options"
)

type Repositorio struct {
	Cliente *mongo.Client
	DB      *mongo.Database
}

func ConectarMongo(ctx context.Context, uri, nombre string) (*Repositorio, error) {
	c, err := mongo.Connect(options.Client().ApplyURI(uri).SetServerSelectionTimeout(2 * time.Second).SetConnectTimeout(2 * time.Second))
	if err != nil {
		return nil, errors.New("configuración de MongoDB inválida")
	}
	if err = c.Ping(ctx, nil); err != nil {
		_ = c.Disconnect(context.Background())
		return nil, errors.New("MongoDB no disponible")
	}
	return &Repositorio{c, c.Database(nombre)}, nil
}

func (r *Repositorio) Preparar(ctx context.Context) error {
	// Validación en servidor además de las validaciones del código Go.
	definiciones := map[string]bson.M{
		"datasets": {"bsonType": "object", "required": bson.A{"_id", "archivo", "fuente", "sha256", "periodo", "vectores"}, "properties": bson.M{
			"_id": bson.M{"bsonType": "string", "minLength": 1}, "archivo": bson.M{"bsonType": "string", "minLength": 1}, "fuente": bson.M{"bsonType": "string", "minLength": 1},
			"sha256": bson.M{"bsonType": "string", "pattern": "^[a-fA-F0-9]{64}$"}, "periodo": bson.M{"enum": bson.A{"1_hora", "24_horas"}}, "vectores": bson.M{"bsonType": bson.A{"int", "long"}, "minimum": 1},
		}},
		"jobs": {"bsonType": "object", "required": bson.A{"_id", "dataset_id", "parametros", "estado", "creado", "actualizado"}, "properties": bson.M{
			"_id": bson.M{"bsonType": "string", "minLength": 1}, "dataset_id": bson.M{"bsonType": "string", "minLength": 1},
			"estado": bson.M{"enum": bson.A{"queued", "running", "completed", "failed"}}, "creado": bson.M{"bsonType": "date"}, "actualizado": bson.M{"bsonType": "date"},
			"parametros": bson.M{"bsonType": "object", "required": bson.A{"k", "semilla", "tolerancia", "max_iter"}}, "resultado": bson.M{"bsonType": "object"}, "error": bson.M{"bsonType": "string"},
		}},
	}
	for nombre, schema := range definiciones {
		validator := bson.M{"$jsonSchema": schema}
		if nombre == "jobs" {
			validator["$or"] = bson.A{bson.M{"estado": "completed", "resultado": bson.M{"$exists": true}}, bson.M{"estado": bson.M{"$ne": "completed"}, "resultado": bson.M{"$exists": false}}}
		}
		err := r.DB.CreateCollection(ctx, nombre, options.CreateCollection().SetValidator(validator))
		var ce mongo.CommandError
		if err != nil && !(errors.As(err, &ce) && ce.Code == 48) {
			return err
		}
	}
	_, err := r.DB.Collection("datasets").Indexes().CreateOne(ctx, mongo.IndexModel{Keys: bson.D{{Key: "sha256", Value: 1}, {Key: "periodo", Value: 1}}, Options: options.Index().SetUnique(true).SetName("dataset_hash_periodo")})
	if err != nil {
		return err
	}
	_, err = r.DB.Collection("jobs").Indexes().CreateMany(ctx, []mongo.IndexModel{
		{Keys: bson.D{{Key: "dataset_id", Value: 1}, {Key: "creado", Value: -1}}, Options: options.Index().SetName("jobs_dataset_fecha")},
		{Keys: bson.D{{Key: "estado", Value: 1}, {Key: "creado", Value: -1}}, Options: options.Index().SetName("jobs_estado_fecha")},
	})
	return err
}
func (r *Repositorio) GuardarDataset(ctx context.Context, d Dataset) error {
	if err := validarDataset(d); err != nil {
		return err
	}
	_, err := r.DB.Collection("datasets").InsertOne(ctx, d)
	return err
}
func (r *Repositorio) Dataset(ctx context.Context, id string) (Dataset, error) {
	var d Dataset
	err := r.DB.Collection("datasets").FindOne(ctx, bson.M{"_id": id}).Decode(&d)
	if errors.Is(err, mongo.ErrNoDocuments) {
		err = ErrNoEncontrado
	}
	return d, err
}
func (r *Repositorio) Crear(ctx context.Context, id, dataset string, p Parametros) error {
	if id == "" {
		return errors.New("job_id vacío")
	}
	d, err := r.Dataset(ctx, dataset)
	if err != nil {
		return err
	}
	if err = validarParametros(p, d.Vectores); err != nil {
		return err
	}
	now := time.Now().UTC()
	_, err = r.DB.Collection("jobs").InsertOne(ctx, Trabajo{ID: id, DatasetID: dataset, Parametros: p, Estado: "queued", Creado: now, Actualizado: now})
	return err
}
func (r *Repositorio) Consultar(ctx context.Context, id string) (Trabajo, error) {
	var j Trabajo
	err := r.DB.Collection("jobs").FindOne(ctx, bson.M{"_id": id}).Decode(&j)
	if errors.Is(err, mongo.ErrNoDocuments) {
		err = ErrNoEncontrado
	}
	return j, err
}
func (r *Repositorio) cambiar(ctx context.Context, id string, previos bson.A, campos bson.M) error {
	campos["actualizado"] = time.Now().UTC()
	u, err := r.DB.Collection("jobs").UpdateOne(ctx, bson.M{"_id": id, "estado": bson.M{"$in": previos}}, bson.M{"$set": campos})
	if err != nil {
		return err
	}
	if u.MatchedCount != 1 {
		return ErrEstado
	}
	return nil
}
func (r *Repositorio) Iniciar(ctx context.Context, id string) error {
	return r.cambiar(ctx, id, bson.A{"queued"}, bson.M{"estado": "running"})
}
func (r *Repositorio) Fallar(ctx context.Context, id, motivo string) error {
	if motivo == "" {
		return errors.New("indique el motivo del fallo")
	}
	return r.cambiar(ctx, id, bson.A{"queued", "running"}, bson.M{"estado": "failed", "error": motivo})
}
func (r *Repositorio) Completar(ctx context.Context, id string, result Resultado) error {
	j, err := r.Consultar(ctx, id)
	if err != nil {
		return err
	}
	d, err := r.Dataset(ctx, j.DatasetID)
	if err != nil {
		return err
	}
	if err = validarResultado(result, j.Parametros, d.Vectores); err != nil {
		return fmt.Errorf("resultado rechazado: %w", err)
	}
	return r.cambiar(ctx, id, bson.A{"running"}, bson.M{"estado": "completed", "resultado": result})
}

## 4. Caché — `redis.go` y `main.go`
Se busca el resultado en Redis y, si no está, se consulta MongoDB. Solo se guardan en caché resultados completados, con una duración de 3 segundos para probar su expiración.

Al cambiar el estado se elimina la clave. Si Redis falla, se consulta MongoDB. `main.go` ejecuta esta demostración.

In [ ]:
%%writefile /content/pc4_persistencia/redis.go
package main

import (
	"context"
	"encoding/json"
	"errors"
	"time"

	"github.com/redis/go-redis/v9"
)

// Solo se cachean resultados completados, que son inmutables. El estado se
// consulta siempre en MongoDB para no mostrar queued/running desactualizados.
type Servicio struct {
	Repo    *Repositorio
	Cache   *redis.Client
	TTL     time.Duration
	Prefijo string
}
type Lectura struct {
	Trabajo Trabajo `json:"trabajo"`
	Fuente  string  `json:"fuente"`
	Aviso   string  `json:"aviso,omitempty"`
}

func NuevaCache(addr, password string) *redis.Client {
	return redis.NewClient(&redis.Options{Addr: addr, Password: password, DB: 0, MaxRetries: -1, DialTimeout: 300 * time.Millisecond, ReadTimeout: 300 * time.Millisecond, WriteTimeout: 300 * time.Millisecond, ContextTimeoutEnabled: true})
}
func (s *Servicio) Clave(id string) string { return s.Prefijo + ":resultado:v1:" + id }
func (s *Servicio) Invalidar(ctx context.Context, id string) error {
	return s.Cache.Del(ctx, s.Clave(id)).Err()
}
func (s *Servicio) Iniciar(ctx context.Context, id string) error {
	if err := s.Repo.Iniciar(ctx, id); err != nil {
		return err
	}
	_ = s.Invalidar(ctx, id)
	return nil
}
func (s *Servicio) Fallar(ctx context.Context, id, motivo string) error {
	if err := s.Repo.Fallar(ctx, id, motivo); err != nil {
		return err
	}
	_ = s.Invalidar(ctx, id)
	return nil
}
func (s *Servicio) Completar(ctx context.Context, id string, r Resultado) error {
	if err := s.Repo.Completar(ctx, id, r); err != nil {
		return err
	}
	_ = s.Invalidar(ctx, id)
	return nil
}
func (s *Servicio) Resultado(ctx context.Context, id string) (Lectura, error) {
	if s.TTL <= 0 {
		return Lectura{}, errors.New("TTL debe ser positivo")
	}
	var j Trabajo
	raw, cacheErr := s.Cache.Get(ctx, s.Clave(id)).Bytes()
	if cacheErr == nil && json.Unmarshal(raw, &j) == nil && j.ID == id && j.Estado == "completed" && j.Resultado != nil {
		return Lectura{Trabajo: j, Fuente: "Redis"}, nil
	}
	j, err := s.Repo.Consultar(ctx, id)
	if err != nil {
		return Lectura{}, err
	}
	if j.Estado != "completed" || j.Resultado == nil {
		return Lectura{Trabajo: j, Fuente: "MongoDB"}, ErrPendiente
	}
	out := Lectura{Trabajo: j, Fuente: "MongoDB"}
	raw, err = json.Marshal(j)
	if err != nil {
		return Lectura{}, err
	}
	if err = s.Cache.Set(ctx, s.Clave(id), raw, s.TTL).Err(); err != nil || (cacheErr != nil && !errors.Is(cacheErr, redis.Nil)) {
		out.Aviso = "Caché no disponible; resultado recuperado de MongoDB"
	}
	return out, nil
}

In [ ]:
%%writefile /content/pc4_persistencia/main.go
package main

import (
	"context"
	"crypto/rand"
	"crypto/sha256"
	"encoding/hex"
	"encoding/json"
	"errors"
	"flag"
	"fmt"
	"os"
	"strconv"
	"time"

	"go.mongodb.org/mongo-driver/v2/bson"
)

func nuevoID() string {
	var b [16]byte
	if _, err := rand.Read(b[:]); err != nil {
		panic(err)
	}
	return hex.EncodeToString(b[:])
}
func hashTexto(s string) string { b := sha256.Sum256([]byte(s)); return hex.EncodeToString(b[:]) }
func requerido(nombre string) string {
	v := os.Getenv(nombre)
	if v == "" {
		panic("Falta variable " + nombre)
	}
	return v
}
func guardar(path string, v any) error {
	b, err := json.MarshalIndent(v, "", "  ")
	if err != nil {
		return err
	}
	return os.WriteFile(path, b, 0600)
}

type ReportePC3 struct {
	Period        string      `json:"period"`
	Vectors       int         `json:"vectors"`
	K             int         `json:"k"`
	Seed          int64       `json:"seed"`
	Tolerance     float64     `json:"tolerance"`
	MaxIterations int         `json:"max_iterations"`
	Centroids     [][]float64 `json:"centroids"`
	Counts        []int       `json:"counts"`
	Iterations    int         `json:"iterations"`
	Inertia       float64     `json:"inertia"`
	Converged     bool        `json:"converged"`
	LoadSeconds   float64     `json:"load_seconds"`
	TrainSeconds  float64     `json:"train_seconds"`
}
type Paso struct {
	Caso      string `json:"caso"`
	Estado    string `json:"estado"`
	Fuente    string `json:"fuente"`
	TTLms     int64  `json:"ttl_ms"`
	Resultado string `json:"resultado"`
}

func ejecutar() error {
	entrada := flag.String("reporte", "", "JSON producido por la referencia PC3")
	metadata := flag.String("dataset", "", "Metadatos del CSV real")
	salida := flag.String("salida", "evidencia.json", "Archivo de evidencia")
	flag.Parse()
	ctx, cancel := context.WithTimeout(context.Background(), 60*time.Second)
	defer cancel()
	r, err := ConectarMongo(ctx, requerido("MONGODB_URI"), requerido("MONGODB_DATABASE"))
	if err != nil {
		return err
	}
	defer r.Cliente.Disconnect(context.Background())
	if err = r.Preparar(ctx); err != nil {
		return err
	}
	c := NuevaCache(requerido("REDIS_ADDR"), os.Getenv("REDIS_PASSWORD"))
	defer c.Close()
	if err = c.Ping(ctx).Err(); err != nil {
		return errors.New("Redis no disponible para la demostración")
	}
	ttl, err := strconv.Atoi(requerido("CACHE_TTL_SECONDS"))
	if err != nil || ttl < 1 || ttl > 30 {
		return errors.New("use TTL de 1 a 30 segundos en la demostración")
	}
	s := &Servicio{r, c, time.Duration(ttl) * time.Second, os.Getenv("MONGODB_DATABASE")}
	var d Dataset
	var a ReportePC3
	b, err := os.ReadFile(*metadata)
	if err != nil {
		return err
	}
	if err = json.Unmarshal(b, &d); err != nil {
		return err
	}
	b, err = os.ReadFile(*entrada)
	if err != nil {
		return err
	}
	if err = json.Unmarshal(b, &a); err != nil {
		return err
	}
	if d.Periodo != a.Period || d.Vectores != a.Vectors {
		return errors.New("reporte y dataset no corresponden")
	}
	// El identificador del dataset depende del contenido y periodo.
	d.ID = d.SHA256 + ":" + d.Periodo
	if previo, e := r.Dataset(ctx, d.ID); errors.Is(e, ErrNoEncontrado) {
		err = r.GuardarDataset(ctx, d)
	} else if e != nil {
		return e
	} else if previo != d {
		return errors.New("metadatos distintos para un mismo dataset")
	}
	if err != nil {
		return err
	}
	p := Parametros{a.K, a.Seed, a.Tolerance, a.MaxIterations}
	result := Resultado{a.Centroids, a.Counts, a.Iterations, a.Inertia, a.Converged, a.LoadSeconds, a.TrainSeconds, "referencia secuencial PC3 ejecutada en este cuaderno; no es clúster PC4"}
	id := "pc4-" + nuevoID()
	if err = r.Crear(ctx, id, d.ID, p); err != nil {
		return err
	}
	pasos := []Paso{}
	pendiente := func(estado string) error {
		v, e := s.Resultado(ctx, id)
		if !errors.Is(e, ErrPendiente) || v.Trabajo.Estado != estado {
			return errors.New("estado pendiente incorrecto")
		}
		if n, _ := c.Exists(ctx, s.Clave(id)).Result(); n != 0 {
			return errors.New("se cacheó un trabajo pendiente")
		}
		pasos = append(pasos, Paso{"Consulta de estado", estado, "MongoDB", -2, "PASS: sin resultado ni caché"})
		return nil
	}
	if err = pendiente("queued"); err != nil {
		return err
	}
	if err = s.Iniciar(ctx, id); err != nil {
		return err
	}
	if err = pendiente("running"); err != nil {
		return err
	}
	if err = s.Completar(ctx, id, result); err != nil {
		return err
	}
	leer := func(caso, fuente string) error {
		v, e := s.Resultado(ctx, id)
		if e != nil {
			return e
		}
		if v.Fuente != fuente {
			return fmt.Errorf("%s: fuente %s, se esperaba %s", caso, v.Fuente, fuente)
		}
		dur, e := c.PTTL(ctx, s.Clave(id)).Result()
		if e != nil {
			return e
		}
		pasos = append(pasos, Paso{caso, v.Trabajo.Estado, v.Fuente, dur.Milliseconds(), "PASS"})
		return nil
	}
	if err = leer("Primera consulta sin caché", "MongoDB"); err != nil {
		return err
	}
	if err = leer("Segunda consulta con caché", "Redis"); err != nil {
		return err
	}
	time.Sleep(s.TTL + 100*time.Millisecond)
	if err = leer("Consulta después de expirar", "MongoDB"); err != nil {
		return err
	}
	if err = s.Invalidar(ctx, id); err != nil {
		return err
	}
	if n, e := c.Exists(ctx, s.Clave(id)).Result(); e != nil || n != 0 {
		return errors.New("clave no invalidada")
	}
	if err = leer("Consulta después de invalidar", "MongoDB"); err != nil {
		return err
	}
	job, err := r.Consultar(ctx, id)
	if err != nil {
		return err
	}
	cacheValue, err := c.Get(ctx, s.Clave(id)).Result()
	if err != nil {
		return err
	}
	indices := map[string][]bson.M{}
	for _, name := range []string{"datasets", "jobs"} {
		cur, e := r.DB.Collection(name).Indexes().List(ctx)
		if e != nil {
			return e
		}
		var list []bson.M
		if e = cur.All(ctx, &list); e != nil {
			return e
		}
		indices[name] = list
	}
	fallido := "pc4-fallo-" + nuevoID()
	if err = r.Crear(ctx, fallido, d.ID, p); err != nil {
		return err
	}
	if err = s.Fallar(ctx, fallido, "fallo de procesamiento simulado para probar almacenamiento"); err != nil {
		return err
	}
	failed, err := r.Consultar(ctx, fallido)
	if err != nil {
		return err
	}
	if err = guardar(*salida, map[string]any{"fecha_utc": time.Now().UTC(), "dataset": d, "job_id": id, "documento_mongodb": job, "ejecucion_fallida": failed, "indices": indices, "clave_redis": s.Clave(id), "valor_redis": json.RawMessage(cacheValue), "pruebas": pasos, "alcance": "A1-A3: persistencia y caché; API y clúster no ejecutados"}); err != nil {
		return err
	}
	for _, paso := range pasos {
		fmt.Printf("%s | %s | %s | %s\n", paso.Caso, paso.Estado, paso.Fuente, paso.Resultado)
	}
	fmt.Printf("Evidencia: %s\njob_id: %s\n", *salida, id)
	return nil
}
func main() {
	if err := ejecutar(); err != nil {
		fmt.Fprintln(os.Stderr, err)
		os.Exit(1)
	}
}

## 5. Pruebas
Se comprueban escritura, consulta, índices, estados, fallos y caché. También se verifica que varias solicitudes no inicien el mismo trabajo.

In [ ]:
%%writefile /content/pc4_persistencia/persistencia_test.go
package main

import (
	"context"
	"errors"
	"math"
	"os"
	"reflect"
	"strings"
	"sync"
	"sync/atomic"
	"testing"
	"time"

	"go.mongodb.org/mongo-driver/v2/bson"
	"go.mongodb.org/mongo-driver/v2/mongo"
)

func ejemplo() (Dataset, Parametros, Resultado) {
	return Dataset{"prueba-4", "fixture_4.csv", "fixture sintético de prueba", strings.Repeat("a", 64), "1_hora", 4},
		Parametros{2, 42, .001, 100}, Resultado{Centroides: [][]float64{{1.5, 1.5}, {10.5, 10.5}}, Conteos: []int{2, 2}, Iteraciones: 2, Inercia: 2, Convergio: true, Origen: "fixture conocido; no es ejecución distribuida"}
}
func TestValidaciones(t *testing.T) {
	d, p, r := ejemplo()
	if validarDataset(d) != nil || validarParametros(p, 4) != nil || validarResultado(r, p, 4) != nil {
		t.Fatal("fixture válido rechazado")
	}
	d.SHA256 = "incorrecto"
	if validarDataset(d) == nil {
		t.Fatal("hash inválido aceptado")
	}
	r.Inercia = math.NaN()
	if validarResultado(r, p, 4) == nil {
		t.Fatal("NaN aceptado")
	}
	_, _, r = ejemplo()
	r.Conteos = []int{1, 2}
	if validarResultado(r, p, 4) == nil {
		t.Fatal("conteos incompletos aceptados")
	}
	p.K = 0
	if validarParametros(p, 4) == nil {
		t.Fatal("K inválido aceptado")
	}
}
func servicioPrueba(t *testing.T) (*Servicio, context.Context) {
	t.Helper()
	if os.Getenv("PC4_INTEGRATION") != "1" {
		t.Skip("PC4_INTEGRATION=1 requiere MongoDB y Redis reales")
	}
	ctx, cancel := context.WithTimeout(context.Background(), 30*time.Second)
	t.Cleanup(cancel)
	r, err := ConectarMongo(ctx, os.Getenv("MONGODB_URI"), os.Getenv("MONGODB_DATABASE"))
	if err != nil {
		t.Fatal(err)
	}
	t.Cleanup(func() { _ = r.Cliente.Disconnect(context.Background()) })
	if err = r.Preparar(ctx); err != nil {
		t.Fatal(err)
	}
	c := NuevaCache(os.Getenv("REDIS_ADDR"), os.Getenv("REDIS_PASSWORD"))
	t.Cleanup(func() { _ = c.Close() })
	if err = c.Ping(ctx).Err(); err != nil {
		t.Fatal(err)
	}
	s := &Servicio{r, c, 2 * time.Second, os.Getenv("MONGODB_DATABASE")}
	return s, ctx
}
func crearFixture(t *testing.T, s *Servicio, ctx context.Context) (string, Resultado) {
	t.Helper()
	d, p, r := ejemplo()
	id := nuevoID()
	d.ID = id
	d.SHA256 = hashTexto(id)
	if err := s.Repo.GuardarDataset(ctx, d); err != nil {
		t.Fatal(err)
	}
	if err := s.Repo.Crear(ctx, id, d.ID, p); err != nil {
		t.Fatal(err)
	}
	return id, r
}
func TestMongoEscrituraConsultaIndices(t *testing.T) {
	s, ctx := servicioPrueba(t)
	id, _ := crearFixture(t, s, ctx)
	j, err := s.Repo.Consultar(ctx, id)
	if err != nil || j.Estado != "queued" {
		t.Fatal(j, err)
	}
	if err = s.Repo.Crear(ctx, id, j.DatasetID, j.Parametros); !mongo.IsDuplicateKeyError(err) {
		t.Fatal("no se rechazó job_id duplicado", err)
	}
	d, err := s.Repo.Dataset(ctx, j.DatasetID)
	if err != nil {
		t.Fatal(err)
	}
	d.ID = nuevoID()
	if err = s.Repo.GuardarDataset(ctx, d); !mongo.IsDuplicateKeyError(err) {
		t.Fatal("índice hash/periodo no rechazó duplicado", err)
	}
	_, err = s.Repo.DB.Collection("jobs").InsertOne(ctx, bson.M{"_id": nuevoID(), "estado": "inventado"})
	if err == nil {
		t.Fatal("MongoDB aceptó documento fuera del esquema")
	}
	_, err = s.Repo.Consultar(ctx, "no-existe")
	if !errors.Is(err, ErrNoEncontrado) {
		t.Fatal(err)
	}
}
func TestEstadosResultadoAtomico(t *testing.T) {
	s, ctx := servicioPrueba(t)
	id, result := crearFixture(t, s, ctx)
	if err := s.Completar(ctx, id, result); !errors.Is(err, ErrEstado) {
		t.Fatal("queued no debe completarse directamente", err)
	}
	if err := s.Iniciar(ctx, id); err != nil {
		t.Fatal(err)
	}
	bad := result
	bad.Conteos = []int{1, 2}
	if s.Completar(ctx, id, bad) == nil {
		t.Fatal("resultado incompleto aceptado")
	}
	j, _ := s.Repo.Consultar(ctx, id)
	if j.Estado != "running" || j.Resultado != nil {
		t.Fatal("se publicó un resultado parcial")
	}
	if err := s.Completar(ctx, id, result); err != nil {
		t.Fatal(err)
	}
	j, err := s.Repo.Consultar(ctx, id)
	if err != nil || j.Estado != "completed" || !reflect.DeepEqual(*j.Resultado, result) {
		t.Fatal(j, err)
	}
	if err = s.Fallar(ctx, id, "fallo tardío"); !errors.Is(err, ErrEstado) {
		t.Fatal("estado terminal mutable", err)
	}
	if err = s.Completar(ctx, id, result); !errors.Is(err, ErrEstado) {
		t.Fatal("resultado terminal mutable", err)
	}
}
func TestEjecucionFallida(t *testing.T) {
	s, ctx := servicioPrueba(t)
	id, _ := crearFixture(t, s, ctx)
	if err := s.Fallar(ctx, id, "nodo desconectado: caso de prueba de almacenamiento"); err != nil {
		t.Fatal(err)
	}
	j, err := s.Repo.Consultar(ctx, id)
	if err != nil || j.Estado != "failed" || j.Resultado != nil || j.Error == "" {
		t.Fatal(j, err)
	}
	if _, err = s.Resultado(ctx, id); !errors.Is(err, ErrPendiente) {
		t.Fatal("se publicó resultado de ejecución fallida")
	}
}
func TestCacheMissHitExpiracionYPersistencia(t *testing.T) {
	s, ctx := servicioPrueba(t)
	id, result := crearFixture(t, s, ctx)
	for _, estado := range []string{"queued", "running"} {
		j, err := s.Resultado(ctx, id)
		if !errors.Is(err, ErrPendiente) || j.Trabajo.Estado != estado {
			t.Fatal(j, err)
		}
		if n, _ := s.Cache.Exists(ctx, s.Clave(id)).Result(); n != 0 {
			t.Fatal("trabajo pendiente cacheado")
		}
		if estado == "queued" {
			if err = s.Iniciar(ctx, id); err != nil {
				t.Fatal(err)
			}
		}
	}
	if err := s.Completar(ctx, id, result); err != nil {
		t.Fatal(err)
	}
	a, err := s.Resultado(ctx, id)
	if err != nil || a.Fuente != "MongoDB" {
		t.Fatal(a, err)
	}
	b, err := s.Resultado(ctx, id)
	if err != nil || b.Fuente != "Redis" || !reflect.DeepEqual(a.Trabajo, b.Trabajo) {
		t.Fatal(b, err)
	}
	ttl, err := s.Cache.PTTL(ctx, s.Clave(id)).Result()
	if err != nil || ttl <= 0 || ttl > s.TTL {
		t.Fatal(ttl, err)
	}
	time.Sleep(s.TTL + 100*time.Millisecond)
	c, err := s.Resultado(ctx, id)
	if err != nil || c.Fuente != "MongoDB" {
		t.Fatal(c, err)
	}
	if err = s.Invalidar(ctx, id); err != nil {
		t.Fatal(err)
	}
	if n, _ := s.Cache.Exists(ctx, s.Clave(id)).Result(); n != 0 {
		t.Fatal("clave no invalidada")
	}
	d, err := s.Resultado(ctx, id)
	if err != nil || d.Fuente != "MongoDB" || !reflect.DeepEqual(d.Trabajo, a.Trabajo) {
		t.Fatal("resultado perdido al vaciar caché", err)
	}
}
func TestRedisNoDisponible(t *testing.T) {
	s, ctx := servicioPrueba(t)
	id, r := crearFixture(t, s, ctx)
	if err := s.Iniciar(ctx, id); err != nil {
		t.Fatal(err)
	}
	if err := s.Completar(ctx, id, r); err != nil {
		t.Fatal(err)
	}
	// Puerto reservado sin servicio: no detiene ni modifica el Redis compartido.
	c := NuevaCache("127.0.0.1:1", "")
	defer c.Close()
	s.Cache = c
	x, err := s.Resultado(ctx, id)
	if err != nil || x.Fuente != "MongoDB" || x.Aviso == "" {
		t.Fatal(x, err)
	}
}
func TestMongoNoDisponible(t *testing.T) {
	if os.Getenv("PC4_INTEGRATION") != "1" {
		t.Skip("prueba de integración")
	}
	ctx, cancel := context.WithTimeout(context.Background(), 500*time.Millisecond)
	defer cancel()
	if _, err := ConectarMongo(ctx, "mongodb://127.0.0.1:1", "pc4_prueba"); err == nil {
		t.Fatal("se esperaba error de conexión")
	}
}
func TestTransicionConcurrente(t *testing.T) {
	s, ctx := servicioPrueba(t)
	id, _ := crearFixture(t, s, ctx)
	var ganadores atomic.Int32
	var wg sync.WaitGroup
	for i := 0; i < 8; i++ {
		wg.Add(1)
		go func() {
			defer wg.Done()
			err := s.Iniciar(ctx, id)
			if err == nil {
				ganadores.Add(1)
			} else if !errors.Is(err, ErrEstado) {
				t.Error(err)
			}
		}()
	}
	wg.Wait()
	if ganadores.Load() != 1 {
		t.Fatalf("transiciones aceptadas: %d", ganadores.Load())
	}
}

In [ ]:
GO_FILES=['modelos.go','mongodb.go','redis.go','main.go','persistencia_test.go']
run(['gofmt','-w',*GO_FILES])
run(['go','mod','tidy'])
run(['go','build','-o','persistencia_pc4','.'])
TEST_LOGS={}
for label,extra in [('Pruebas',[]),('Detector de carreras',['-race'])]:
    print('\n'+label)
    command=['go','test','-v','-count=1','-timeout=120s',*extra,*GO_FILES]
    TEST_LOGS[label]={'fecha_utc':datetime.now(timezone.utc).isoformat(),'comando':command,'salida':run(command)}
print('Pruebas y detector de carreras terminados sin errores.')

## 6. Datos y resultados de K-Means
Se utilizan los dos CSV existentes en Drive: 1 hora y 24 horas. La referencia secuencial de la PC3 genera los resultados que se guardarán; la integración con el clúster de la PC4 queda pendiente.

Parámetros: K = 4, semilla = 42, tolerancia = 0.001 y máximo de 100 iteraciones.

In [ ]:
import urllib.request
SOURCES={
 '1_hora':('1oHqxofkw5IXPefc8BDJVM8Z5EfcKE024',445347,'2996aeea838e11ebb0613034ceabd7a51c0a7df9ba1a83f8657f908bfb267204'),
 '24_horas':('10XM6ZWiFNhnzNk13wRhZY5TqkQEReczD',18297,'5866449404c20a23e52b65f94871a365c7979b1fcf0abd7ed91554d70ea609a6')
}
DATASETS={}
for period,(file_id,n,expected_hash) in SOURCES.items():
    url=f'https://drive.google.com/uc?export=download&id={file_id}'
    with urllib.request.urlopen(url,timeout=120) as response: payload=response.read()
    digest=hashlib.sha256(payload).hexdigest()
    assert digest==expected_hash, f'{period}: descarga distinta del CSV verificado; no continuar.'
    path=WORK/f'features_{period}.csv'
    path.write_bytes(payload)
    frame=pd.read_csv(path)
    assert list(frame.columns)==['PM10','PM2.5'] and len(frame)==n
    meta={'dataset_id':digest+':'+period,'archivo':path.name,'fuente':f'https://drive.google.com/file/d/{file_id}/view','sha256':digest,'periodo':period,'vectores':n}
    (WORK/f'dataset_{period}.json').write_text(json.dumps(meta),encoding='utf-8')
    DATASETS[period]=meta
    del frame,payload
display(pd.DataFrame(DATASETS.values())[['archivo','periodo','vectores']])
REFERENCE = WORK/'referencia_breve'
REFERENCE.mkdir(exist_ok=True)

In [ ]:
%%writefile /content/pc4_persistencia/referencia_breve/referencia.go
package main

import (
    "encoding/csv"
    "encoding/json"
    "fmt"
    "io"
    "log"
    "math"
    "math/rand"
    "os"
    "strconv"
    "time"
)

// Dimensions representa la cantidad de características
// que utilizaremos para cada observación: PM10 y PM2.5.
const Dimensions = 2

// Dataset almacena los vectores que utilizaremos en K-Means.
type Dataset struct {
	Points [][]float64
	N      int
	D      int
}

// NewDataset crea un conjunto de datos a partir de una lista de puntos.
func NewDataset(points [][]float64) Dataset {
	return Dataset{
		Points: points,
		N:      len(points),
		D:      Dimensions,
	}
}

// LoadCSV lee un archivo CSV con las columnas PM10 y PM2.5.
func LoadCSV(path string) (Dataset, error) {
	file, err := os.Open(path)
	if err != nil {
		return Dataset{}, fmt.Errorf("no se pudo abrir el archivo: %w", err)
	}
	defer file.Close()

	reader := csv.NewReader(file)

	header, err := reader.Read()
	if err != nil {
		return Dataset{}, fmt.Errorf("no se pudo leer el encabezado: %w", err)
	}

	if len(header) != 2 || header[0] != "PM10" || header[1] != "PM2.5" {
		return Dataset{}, fmt.Errorf("formato de CSV incorrecto: se esperaban PM10 y PM2.5")
	}

	points := make([][]float64, 0)

	for row := 2; ; row++ {
		record, err := reader.Read()

		if err == io.EOF {
			break
		}

		if err != nil {
			return Dataset{}, fmt.Errorf("error al leer la fila %d: %w", row, err)
		}

		if len(record) != 2 {
			return Dataset{}, fmt.Errorf("la fila %d debe tener exactamente 2 columnas", row)
		}

		pm10, err := strconv.ParseFloat(record[0], 64)
		if err != nil {
			return Dataset{}, fmt.Errorf("valor PM10 inválido en la fila %d: %w", row, err)
		}

		pm25, err := strconv.ParseFloat(record[1], 64)
		if err != nil {
			return Dataset{}, fmt.Errorf("valor PM2.5 inválido en la fila %d: %w", row, err)
		}

		if math.IsNaN(pm10) || math.IsInf(pm10, 0) ||
			math.IsNaN(pm25) || math.IsInf(pm25, 0) {
			return Dataset{}, fmt.Errorf("se encontró un valor no finito en la fila %d", row)
		}

		points = append(points, []float64{pm10, pm25})
	}

	if len(points) == 0 {
		return Dataset{}, fmt.Errorf("el CSV no contiene datos")
	}

	return NewDataset(points), nil
}

// initializeCentroids selecciona K puntos aleatorios del dataset
// para utilizarlos como centroides iniciales.
func initializeCentroids(data Dataset, k int, seed int64) ([][]float64, error) {
	if err := validateDataset(data); err != nil {
		return nil, err
	}

	if k <= 0 || k > data.N {
		return nil, fmt.Errorf("K debe estar entre 1 y N")
	}

	rng := rand.New(rand.NewSource(seed))
	indices := rng.Perm(data.N)[:k]

	centroids := make([][]float64, k)

	for i, index := range indices {
		centroids[i] = make([]float64, data.D)
		copy(centroids[i], data.Points[index])
	}

	return centroids, nil
}

// validateDataset comprueba que el conjunto de datos tenga
// una estructura válida para trabajar con K-Means.
func validateDataset(data Dataset) error {
	if data.N <= 0 || len(data.Points) == 0 {
		return fmt.Errorf("el conjunto de datos está vacío")
	}

	if data.N != len(data.Points) {
		return fmt.Errorf("la cantidad de puntos no coincide con N")
	}

	if data.D != 2 {
		return fmt.Errorf("el dataset debe tener exactamente 2 dimensiones")
	}

	for i, point := range data.Points {
		if len(point) != data.D {
			return fmt.Errorf("el punto %d tiene dimensiones inconsistentes", i)
		}

		for _, value := range point {
			if math.IsNaN(value) || math.IsInf(value, 0) {
				return fmt.Errorf("el punto %d contiene un valor no finito", i)
			}
		}
	}

	return nil
}

// squaredDistance calcula la distancia euclidiana al cuadrado
// entre dos puntos.
func squaredDistance(a, b []float64) float64 {
	var distance float64

	for i := range a {
		diff := a[i] - b[i]
		distance += diff * diff
	}

	return distance
}

// assignClusters asigna cada punto al clúster cuyo centroide
// se encuentra más cerca.
func assignClusters(data Dataset, centroids [][]float64) []int {
	labels := make([]int, data.N)

	for i, point := range data.Points {
		bestCluster := 0
		minDistance := squaredDistance(point, centroids[0])

		for j := 1; j < len(centroids); j++ {
			distance := squaredDistance(point, centroids[j])

			if distance < minDistance {
				minDistance = distance
				bestCluster = j
			}
		}

		labels[i] = bestCluster
	}

	return labels
}

// updateCentroids actualiza los centroides calculando el promedio
// de los puntos asignados a cada clúster.
// Si un clúster queda vacío, conserva su centroide anterior.
func updateCentroids(data Dataset, labels []int, centroids [][]float64) [][]float64 {
	k := len(centroids)

	newCentroids := make([][]float64, k)
	counts := make([]int, k)

	for i := range newCentroids {
		newCentroids[i] = make([]float64, data.D)
	}

	// Acumulamos los valores de cada dimensión por clúster.
	for i, point := range data.Points {
		cluster := labels[i]
		counts[cluster]++

		for d := 0; d < data.D; d++ {
			newCentroids[cluster][d] += point[d]
		}
	}

	// Calculamos el promedio de cada clúster.
	for cluster := 0; cluster < k; cluster++ {
		if counts[cluster] > 0 {
			for d := 0; d < data.D; d++ {
				newCentroids[cluster][d] /= float64(counts[cluster])
			}
		} else {
			copy(newCentroids[cluster], centroids[cluster])
		}
	}

	return newCentroids
}

// KMeansResult almacena los resultados obtenidos al ejecutar K-Means.
type KMeansResult struct {
	Centroids  [][]float64
	Labels     []int
	Inertia    float64
	Iterations int
	Converged  bool
}

// calculateInertia calcula la suma de las distancias al cuadrado
// entre cada punto y el centroide de su clúster.
func calculateInertia(data Dataset, centroids [][]float64, labels []int) float64 {
	var inertia float64

	for i, point := range data.Points {
		inertia += squaredDistance(point, centroids[labels[i]])
	}

	return inertia
}

// RunKMeans ejecuta el algoritmo K-Means secuencial.
func RunKMeans(data Dataset, k int, maxIter int, tol float64, seed int64) (KMeansResult, error) {
	if err := validateDataset(data); err != nil {
		return KMeansResult{}, err
	}

	if k <= 0 || k > data.N {
		return KMeansResult{}, fmt.Errorf("K debe estar entre 1 y N")
	}

	if maxIter <= 0 {
		return KMeansResult{}, fmt.Errorf("el máximo de iteraciones debe ser positivo")
	}

	if tol < 0 || math.IsNaN(tol) || math.IsInf(tol, 0) {
		return KMeansResult{}, fmt.Errorf("la tolerancia debe ser finita y no negativa")
	}

	centroids, err := initializeCentroids(data, k, seed)
	if err != nil {
		return KMeansResult{}, err
	}

	result := KMeansResult{}

	for iteration := 0; iteration < maxIter; iteration++ {
		labels := assignClusters(data, centroids)
		newCentroids := updateCentroids(data, labels, centroids)

		var movement float64

		for cluster := 0; cluster < k; cluster++ {
			movement += squaredDistance(centroids[cluster], newCentroids[cluster])
		}

		centroids = newCentroids
		result.Iterations = iteration + 1

		if math.Sqrt(movement) < tol {
			result.Converged = true
			break
		}
	}

	// Reasignamos los puntos usando los centroides finales.
	result.Centroids = centroids
	result.Labels = assignClusters(data, centroids)
	result.Inertia = calculateInertia(data, centroids, result.Labels)

	return result, nil
}

// Ejecutamos la referencia y guardamos el resumen que recibirá MongoDB.
func main() {
    inicio := time.Now()
    datos, err := LoadCSV(os.Args[1])
    if err != nil { log.Fatal(err) }
    carga := time.Since(inicio).Seconds()

    inicio = time.Now()
    resultado, err := RunKMeans(datos, 4, 100, 0.001, 42)
    if err != nil { log.Fatal(err) }
    duracion := time.Since(inicio).Seconds()
    conteos := make([]int, 4)
    for _, grupo := range resultado.Labels { conteos[grupo]++ }

    resumen := map[string]any{
        "period": os.Args[2], "vectors": datos.N,
        "k": 4, "seed": 42, "tolerance": 0.001, "max_iterations": 100,
        "centroids": resultado.Centroids, "counts": conteos,
        "iterations": resultado.Iterations, "inertia": resultado.Inertia,
        "converged": resultado.Converged,
        "load_seconds": carga, "train_seconds": duracion,
    }
    if err := json.NewEncoder(os.Stdout).Encode(resumen); err != nil { log.Fatal(err) }
}

In [ ]:
run(['go', 'build', '-o', 'referencia_pc3_bin', str(REFERENCE/'referencia.go')])
MODEL_REPORTS = {}
for period in DATASETS:
    salida = subprocess.check_output([
        './referencia_pc3_bin', str(WORK/f'features_{period}.csv'), period
    ], text=True)
    report = json.loads(salida)
    assert report['vectors'] == DATASETS[period]['vectores']
    assert sum(report['counts']) == report['vectors']
    (WORK/f'reporte_{period}.json').write_text(salida)
    MODEL_REPORTS[period] = report

display(pd.DataFrame([
    {'Periodo': p, 'Vectores': r['vectors'], 'Iteraciones': r['iterations'],
     'Convergió': r['converged'], 'Inercia': r['inertia']}
    for p, r in MODEL_REPORTS.items()
]))


## 7. Ejecución y resultados
Para cada periodo se guarda el resultado en MongoDB y se consulta con y sin caché. Se muestran el documento, los índices y las pruebas de expiración e invalidación.

In [ ]:
EVIDENCES = {}
for period in DATASETS:
    print('PERIODO:', period)
    run([str(WORK/'persistencia_pc4'), '-dataset', str(WORK/f'dataset_{period}.json'),
         '-reporte', str(WORK/f'reporte_{period}.json'), '-salida', str(WORK/f'evidencia_{period}.json')])
    e = json.loads((WORK/f'evidencia_{period}.json').read_text())
    EVIDENCES[period] = e
    display(pd.DataFrame(e['pruebas']))
    print('Documento MongoDB:', json.dumps(e['documento_mongodb'], ensure_ascii=False))
    print('Clave Redis:', e['clave_redis'])
    print('Índices:', json.dumps(e['indices'], ensure_ascii=False))

## 8. Resumen
Se verifica que los centroides, conteos, inercia e iteraciones recuperados coincidan con el resultado original. La integración con la API y el clúster se realiza después con el equipo.

In [ ]:
assert len(EVIDENCES) == 2
for period, evidence in EVIDENCES.items():
    stored = evidence['documento_mongodb']['resultado']
    original = MODEL_REPORTS[period]
    assert stored['centroides'] == original['centroids']
    assert stored['conteos'] == original['counts']
    assert stored['inercia'] == original['inertia']
    assert stored['iteraciones'] == original['iterations']

display(pd.DataFrame([
    {'Periodo': p, 'Vectores': e['dataset']['vectores'],
     'Estado': e['documento_mongodb']['estado'], 'Persistencia y caché': 'PASS'}
    for p, e in EVIDENCES.items()
]))
print('Resultados guardados y recuperados correctamente.')

## Referencias
- [MongoDB en Go](https://www.mongodb.com/docs/drivers/go/current/)
- [Redis en Go](https://redis.io/docs/latest/develop/clients/go/)